# Module 1: Path Extraction and Encoding

### The concept

An I-JEPA encoder (like most ViTs) doesn't operate on pixels directly. It first **chops the image into a grid of patches**, then **linearly projects each patch into a vector**. From that point on, the model never sees pixels again — it only sees a sequence of vectors.

Here's the target transformation:

$$
\underbrace{x}_{\text{image}} \in \mathbb{R}^{B \times C \times H \times W} \xrightarrow{\text{patchify}} \underbrace{x_p}_{\text{flat patches}} \in \mathbb{R}^{B \times N \times (P^2 \cdot C)} \xrightarrow{W_e} \underbrace{z}_{\text{token embeddings}} \in \mathbb{R}^{B \times N \times D}
$$

where $N = \frac{H}{P} \cdot \frac{W}{P}$ is the number of patches and $P$ is the patch size.

With our tiny model: $B=2,\ C=1,\ H=W=16,\ P=4 \Rightarrow N=16,\ D=16$

---

### 🤔 Pre-coding questions — answer these before touching the keyboard

**Q1.** In our tiny model, after patchifying a single image (`B=1`), you have 16 patches. Each patch is a 4×4 region of a 1-channel image. What is the length of the flattened vector representing one patch? How did you get there?

- Answer: 16 patches come from  the $N = \frac{H}{P} \cdot \frac{W}{P} = \frac{16}{4} \cdot \frac{16}{4} = 16$ so we can imagine from a 16x16 grid, we have $N=16, \ 1\times4\times4$ ( $C\times P \times P$ ) patches. If we vectorize it we have a $1\times16$ or simply a $16$ element patch vector (i.e., $x_p \in \mathbb{R}^{B\times N \times (P^2 \cdot C)}$). 
    

**Q2.** The embedding matrix $W_e \in \mathbb{R}^{P^2 C \times D}$ is just a linear layer with no bias (for now). If you matrix-multiply your 16 flattened patches against $W_e$, what is the exact shape of the result for a single image? What about for a batch of 2?

- Answer: To recall, $D$ is the embedding vector dimension. So, if we have an embedding projection $W_e \in \mathbb{R}^{B \times (P^2 \cdot C) \times D}$ which converts the $x_p$ into $z_i=W_ex_p$. So the exact shape for $B=1$ would be $D=16$ (the example was just coincidental). If $B=2$ then we have $z \in \mathbb{R}^{B \times N \times D}$ . Don’t forget the $N$ patches.
    

**Q3.** When you reshape the image into patches, you need to go from shape `(B, C, H, W)` → `(B, N, P, P, C)` → `(B, N, P²C)`. What NumPy operations would you use to do this? (Don't write code yet — just name the operations and their order.)

- Answer: I would use the np.reshape  most of the time. 1st do `np.reshape` and transposes to convert (B,C,H,W) into (B,N,P,P,C) then of course another reshape to convert to (B,N,P2C).
    

**Q4.** This one is more conceptual: after patch embedding, **all spatial 2D structure is gone** — the model just sees a flat list of $N$ vectors. Does that concern you? What information has been lost, and what hasn't?

- Answer: I think the global spatial 2D information is gone but rather in small chunks. So the local information in those small chunks are still together. In other words the spatial information within the 4x4 are still existing. However the correlated information between two adjacent patches is lost.
    

**Q5.** The same $W_e$ is applied to **every** patch identically. What does that imply about how the model treats patches at different positions at this stage? 

- Answer: I guess it implies spatial invariance where the relative position in space does not matter. That information is not relevant. For example  a patch of sky in the top-left corner and an identical patch of sky in the bottom-right corner produce **bit-for-bit identical embeddings**. The encoder cannot tell them apart yet.

# Coding Exercise

In [1]:
import numpy as np

# ── Tiny model constants ──────────────────────────────────────
B, C, H, W  = 2, 1, 16, 16
P           = 4          # patch size
N           = (H // P) * (W // P)   # 16 patches
D           = 16         # embedding dim

# Random image batch
x = np.random.randn(B, C, H, W)

# Embedding matrix  (no bias for now)
We = np.random.randn(P * P * C, D) * 0.02

# ── Your job: fill these two functions ───────────────────────

def patchify(x, P):
    """
    Input:  x  of shape (B, C, H, W)
    Output: xp of shape (B, N, P*P*C)

    Hint: you will need at least one reshape AND one transpose.
    Print intermediate shapes as you go.
    """
    # Get input dimensions
    B, C, H, W = x.shape
    # Patch sizes
    h_patches = H // P
    w_patches = W // P
    # Reshape (B, C, H, W) to (B, C, h_patches, P, w_patches, P)
    x_reshaped = x.reshape(B, C, h_patches, P, w_patches, P)
    # Transpose to (B, h_patches, w_patches, P, P, C)
    x_transposed = x_reshaped.transpose(0, 2, 4, 3, 5, 1)
    # Reshape (B, h_patches, w_patches, P, P, C) to (B, N, P*P*C)
    xp = x_transposed.reshape(B, N, P*P*C)
    return xp

def patch_embed(xp, We):
    """
    Input:  xp of shape (B, N, P*P*C)
            We of shape (P*P*C, D)
    Output: z  of shape (B, N, D)

    Hint: a single np.matmul call is enough.
    """
    return np.matmul(xp, We)

# ── Shape checks (these should pass when you're done) ─────────
xp = patchify(x, P)
assert xp.shape == (B, N, P*P*C), f"Got {xp.shape}"

z = patch_embed(xp, We)
assert z.shape == (B, N, D), f"Got {z.shape}"

print("xp:", xp.shape)   # expect (2, 16, 16)
print("z: ", z.shape)    # expect (2, 16, 16)

xp: (2, 16, 16)
z:  (2, 16, 16)


# Special Notes
- Take note that the $(P,P,C)$ ordering was intentional and it should NOT be in the form of $(C,P,P)$ nor $(P,C,P)$.
- After flattening, your patch vector looks like this with C last (P, P, C):

```text
        [ pixel(0,0)_c0, pixel(0,0)_c1, pixel(0,1)_c0, pixel(0,1)_c1, ... ]
```

- Now imagine C got sandwiched in the middle, giving (P, C, P) before flattening:

```text
        [ pixel(0,0)_c0, pixel(0,0)_c1 ... pixel(1,0)_c0 ... ]
```

- Wait — that actually looks the same here since C=1. So let's make it concrete: what if C=3 (RGB)?
- With (P, P, C) flattened, contiguous memory gives you:

```text
        [ R(0,0), G(0,0), B(0,0),   R(0,1), G(0,1), B(0,1),   ... ]
        └── pixel (0,0) ──┘         └── pixel (0,1) ──┘
```

- With (P, C, P) flattened, you'd get:

```text
        [ R(0,0), G(0,0), B(0,0),   R(0,0)_row2?? ... ]
```

- It scrambles. Each row of $W_e$​ would be learning from a mixed spatial + channel index that has no clean geometric meaning.
- Each row of $W_e$​ corresponds to **one input feature** — which, with correct `(P, P, C)` ordering, means one specific (spatial position, channel) pair. So $W_e$​ is learning: *"how much does pixel (i,j)'s red/green/blue value contribute to each of the D output dimensions?"*